# Two_Breaks — LLM review notebook (COMMENTARY ONLY)

**This notebook does not analyse data.** It reads finished products and asks an
Anthropic model to comment on them. It is deliberately separate from
`Two_Breaks_single_GRB_pipeline.ipynb`, which is the *trust anchor*: pure
threeML/scripts computation, no LLM, no network.

PI ruling (2026-08-17): *"I want to trust these fits and that none is produced
or hallucinated data; so maybe we keep only the fitting part in notebooks?"*

Guarantees enforced in `grb_llm.py`:
1. **Quote-only numbers** — the model may not state a number absent from the
   supplied context; every response is scanned and unquoted numeric tokens are
   flagged in the output header.
2. **Quarantine** — output goes to `notebooks/llm_review/<trig>/`, banner-marked
   AI-GENERATED, with a provenance sidecar (model, prompt hash, token usage).
   Nothing under `results/` is ever written.
3. **Fail-soft, never fake** — no key or no network returns `None` with a message.

Requires `ANTHROPIC_API_KEY` in the project `.env`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))
import grb_llm

BURST = os.environ.get("GRB_BURST", "bn090530760")
ROOT = os.path.dirname(os.path.abspath("."))

# the trust anchor must stay LLM-free — fails loudly if that ever changes
grb_llm.assert_pipeline_is_pure()
print("burst:", BURST, "| model:", grb_llm.MODEL)
print("key present:", bool(grb_llm._load_key()))

## 1. Model-census commentary
Reads the per-bin all-model AIC table and comments on decisiveness, validity, and coherence.

In [ ]:
tbl_path = f"{ROOT}/results/convention_check/sed_grid_{BURST}/tables/ALL_MODELS_TABLES.md"
tbl = open(tbl_path).read() if os.path.exists(tbl_path) else ""
print(f"context: {len(tbl)} chars from {os.path.basename(tbl_path)}")
out = grb_llm.review_model_census(tbl[:40000], BURST)
print(out if out else "(no commentary produced)")

## 2. Temporal commentary
Estimator-labelled temporal values; asks whether estimators disagree beyond quoted errors and whether caveats travel with the numbers.

In [ ]:
import json as _json
from astropy.table import Table
parts = []
tc = Table.read(f"{ROOT}/results/temporal_catalog_all106.ecsv")
tcol = [c for c in tc.colnames if "TRIG" in c.upper()][0]
row = tc[[str(x).strip() == BURST for x in tc[tcol]]]
if len(row):
    keep = [c for c in tc.colnames if any(k in c.upper() for k in ("T90","T50","MVT","LAG","GOWRI"))]
    parts.append("TEMPORAL CATALOG ROW:\n" + "\n".join(f"{c} = {row[0][c]}" for c in keep))
for name in ("step7_figs", "step7_lag_latbright"):
    p = f"{ROOT}/results/sweep106/{BURST}/{BURST}_{name}.json"
    if os.path.exists(p):
        parts.append(f"{name}.json:\n" + _json.dumps(_json.load(open(p)), indent=1, default=str))
p = f"{ROOT}/results/mvt_upstream/run_step7/{BURST}/result.json"
if os.path.exists(p):
    parts.append("canonical MVT (Bala) result.json:\n" + _json.dumps(_json.load(open(p)), indent=1, default=str))
summary = "\n\n".join(parts)
out = grb_llm.review_temporal(summary, BURST)
print(out if out else "(no commentary produced)")

## 3. Figure commentary (vision)
Sends a figure with the standing contract and its product context; asks whether the figure shows what the context claims.

In [ ]:
import glob
contract = open(f"{ROOT}/dev/ai_guides/FigureVisionQC.md").read()[:8000]
png = sorted(glob.glob(f"{ROOT}/results/sweep106/{BURST}/{BURST}_step9_qc.png"))
if png:
    ctx = tbl[:20000] if tbl else "(no table context available)"
    out = grb_llm.review_figure(png[0], contract, ctx, BURST, "step9_figure_commentary")
    print(out if out else "(no commentary produced)")
else:
    print("no step9 figure for this burst")

## 4. Where the output lives

`notebooks/llm_review/<trig>/*.md` (banner-marked) and `*.json` (provenance).
Nothing here feeds back into `results/` or into any paper's numbers — papers
quote products only. Any numeric token the model produced that was not present
in its context appears as a **GUARD FLAG** in the file header.

In [ ]:
import glob
for p in sorted(glob.glob(f"{ROOT}/notebooks/llm_review/{BURST}/*")):
    print(f"{os.path.getsize(p):8d}  {p.split('llm_review/')[-1]}")